In [1]:
# 필요한 LlamaIndex, Cohere, PDF Reader 패키지를 설치하는 안내 셀입니다.
# llamaIndex로 연결할 Cohere / PDF Reader 설치
# - python-dotenv: .env 파일에서 COHERE_API_KEY를 읽기 위해 사용합니다.
# !pip install llama-index-llms-cohere
# !pip install llama-index-embeddings-cohere
# !pip install llama-index-readers-file
# !pip install pypdf
# !pip install python-dotenv

In [2]:
# 파일 경로, 환경 변수, Cohere 모델, LlamaIndex 구성 요소를 불러옵니다.
import os
from dotenv import load_dotenv, find_dotenv

from llama_index.readers.file import PDFReader
from llama_index.llms.cohere import Cohere
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader

In [3]:
# PDF 데이터 경로와 Cohere API 설정을 준비합니다.
# Cohere API Key는 Git에 올리지 않기 위해 .env 파일에서 읽습니다.
# 프로젝트 루트에 `.env` 파일을 만들고 아래처럼 저장하세요.
# COHERE_API_KEY=your_cohere_api_key
env_path = find_dotenv(filename='.env', usecwd=True)
load_dotenv(env_path)

cohere_api_key = os.getenv('COHERE_API_KEY')
if not cohere_api_key:
    raise ValueError('COHERE_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요.')

llm = Cohere(
    model='command-r-08-2024',
    # model='command-r7b-l2-2024',
    api_key=cohere_api_key,
    temperature=0,  # 낮을수록 일관된 답변을 생성합니다.
)
embed_model = CohereEmbedding(
    api_key=cohere_api_key,
    model_name='embed-multilingual-v3.0',
    input_type='search_document',
)

Settings.llm = llm
Settings.embed_model = embed_model

print('Cohere 설정 완료')

Cohere 설정 완료


In [4]:
# 지정한 폴더의 PDF 파일을 LlamaIndex Document 형태로 읽어옵니다.
documents = SimpleDirectoryReader(
    input_dir='../Data/pdf_sample1',
    file_extractor={'.pdf' : PDFReader()}
).load_data()

In [5]:
# 읽어온 문서 개수를 확인합니다.
print(f'로드된 문서 수 : {len(documents)}')

로드된 문서 수 : 23


In [6]:
# 첫 번째 문서 내용을 출력해 로드 결과를 확인합니다.
print(documents[0])

Doc ID: 5eb38256-1dd1-47e3-b053-968d53269caa
Text: 2024 미국의 인공지능(AI) 정책․전략 현황과 변화 방향     - AI 지배력 강화와 초강대국 유지를 위해
미국은 어떻게 변화하고 있는가?-


In [7]:
# 문서 임베딩을 생성하고 벡터 인덱스를 만듭니다.
index = VectorStoreIndex.from_documents(documents) # vector database 만든거

2026-06-02 11:38:55,982 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:38:56,351 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:38:56,803 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:38:57,232 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:38:57,578 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"


In [8]:
# 생성한 인덱스를 질의 엔진으로 변환합니다.
query_engine = index.as_query_engine()

In [9]:
# 질의 엔진에 질문을 보내 응답을 생성합니다.
response = query_engine.query('너를 이용해 trpg 마스터 ai를 개발하고 싶은데 어떤 데이터가 필요하니?')

2026-06-02 11:38:57,816 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:06,903 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


In [10]:
# 생성된 최종 응답을 출력합니다.
print(response)

To develop a TRPG master AI, you would need a diverse and comprehensive dataset that covers various aspects of tabletop role-playing games (TRPGs). Here's a list of potential data sources and types that could be beneficial:

1. TRPG Rulebooks:
   - Obtain rulebooks and game manuals from different TRPG systems. These books contain the core mechanics, character creation rules, combat systems, and world-building guidelines.
   - Extract and organize data from these rulebooks, including character attributes, skills, abilities, and game mechanics.

2. Character Profiles:
   - Collect detailed character profiles from various TRPG campaigns and adventures.
   - Include information such as character backgrounds, personalities, strengths, weaknesses, and unique abilities.

3. Adventure Logs and Campaign Notes:
   - Gather logs and notes from past TRPG campaigns, including session summaries, plot points, and player decisions.
   - Analyze these logs to understand the flow of the game, player int

In [11]:
# 응답 생성에 사용된 근거 문서의 metadata와 score를 확인합니다.
for node in response.source_nodes:
    print(node.score)

0.5072000681369911
0.4841651452015795
